# Load libraries

In [ ]:
!pip install tensorflow-text

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
import tensorflow_text as text
import tf_keras
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Load & Preprocess Dataset

In [ ]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.rename(columns = {'v1':'Category', 'v2':'Message'}, inplace = True)
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
# df.groupby('Category').describe()

Message                                                            \
           count unique                                                top   
Category                                                                     
ham         4825   4516                             Sorry, I'll call later   
spam         747    641  Please call our customer service representativ...   

               
         freq  
Category       
ham        30  
spam        4

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
encoded_array = encoder.fit_transform(df[['Category']])

one_hot_encoded_df_sklearn = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(['Category']))

df = pd.concat([df,one_hot_encoded_df_sklearn], axis=1)
df.head()


,Category,Message,Category_ham,Category_spam,Category_ham,Category_spam
0,ham,"Go until jurong point, crazy.. Available only ...",1.0,0.0,1.0,0.0
1,ham,Ok lar... Joking wif u oni...,1.0,0.0,1.0,0.0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,0.0,1.0,0.0,1.0
3,ham,U dun say so early hor... U c already then say...,1.0,0.0,1.0,0.0
4,ham,"Nah I don't think he goes to usf, he lives aro...",1.0,0.0,1.0,0.0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['Message'],one_hot_encoded_df_sklearn)

In [ ]:
X_train.head()

,Message
3184,Dunno i juz askin cos i got a card got 20% off...
2348,But i dint slept in afternoon.
5045,"Dunno, my dad said he coming home 2 bring us o..."
1868,Mmmm ... Fuck ... Not fair ! You know my weakn...
881,Reminder: You have not downloaded the content ...


In [ ]:
y_train.head()

,Category_ham,Category_spam
3184,1.0,0.0
2348,1.0,0.0
5045,1.0,0.0
1868,1.0,0.0
881,0.0,1.0


# Importing Keras Layers

In [ ]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

# Model Creation

In [ ]:
# def get_sentence_embeding(sentences):
#     preprocessed_text = bert_preprocess(sentences)
#     return bert_encoder(preprocessed_text)['pooled_output']

## Input layer

In [ ]:
text_input = tf_keras.layers.Input(shape=(), dtype=tf.string, name='text')

## Bert layers

In [ ]:
preprocessed_text = bert_preprocess(text_input)

In [ ]:
outputs = bert_encoder(preprocessed_text)

## Neural network layers

In [ ]:
l = tf_keras.layers.Dropout(0.1, name="dropout")(outputs['pooled_output'])
l = tf_keras.layers.Dense(2, activation='softmax', name="output")(l)

# Construct final model

In [ ]:
model = tf_keras.Model(inputs=[text_input], outputs = [l])

In [ ]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 keras_layer (KerasLayer)    {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
                                                                                              

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'],
              run_eagerly=True)

# Training & Evaluation

In [ ]:
model.fit(X_train, y_train, epochs=2, batch_size=32)

Epoch 1/2
131/131 [==============================] - 126s 793ms/step - loss: 0.2708 - accuracy: 0.8911
Epoch 2/2
131/131 [==============================] - 106s 806ms/step - loss: 0.1861 - accuracy: 0.9294


In [ ]:
model.evaluate(X_test, y_test)

#predict
y_predicted = model.predict(X_test)
# y_predicted = y_predicted.flatten()
print(y_predicted)

44/44 [==============================] - 14s 315ms/step
[[0.9605853  0.03941465]
 [0.7086232  0.29137683]
 [0.95993906 0.04006089]
 ...
 [0.8523216  0.14767839]
 [0.7646402  0.23535977]
 [0.96782607 0.03217393]]


In [ ]:
# y_predicted = np.where(y_predicted > 0.5, 1, 0)
# y_predicted

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Convert one-hot encoded targets and probability predictions to class indices (0 for ham, 1 for spam)
y_test_labels = np.argmax(y_test.values, axis=1)
y_pred_labels = np.argmax(y_predicted, axis=1)

print(f"Accuracy Score: {accuracy_score(y_test_labels, y_pred_labels)}")

Accuracy Score: 0.95908111988514


In [ ]:
f1_score(y_test_labels,y_pred_labels)

0.8463611859838275

In [ ]:
print(classification_report(y_test_labels,y_pred_labels))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98      1197
           1       0.90      0.80      0.85       196

    accuracy                           0.96      1393
   macro avg       0.93      0.89      0.91      1393
weighted avg       0.96      0.96      0.96      1393



In [ ]:
model.save('spam_bert_model.h5')

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
